# Inventory Analysis Capstone

Start by importing all necessary Python libraries

In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import requests
import ast
import psycopg2
import sqlalchemy


### Data Acquisition - Inventory


Read in the dataset with the 500 companies first

In [22]:
companies = pd.read_csv('data/SP500.csv')
companies.head(5)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373


Collect Net Inventory from SEC filings for each company

In [45]:
def fetch_sec_data(cik):
    headers = {'User-Agent' : 'marshalln@etown.edu', 'Host': 'data.sec.gov'}
    cik = str(cik).zfill(10)
    url = 'https://data.sec.gov/api/xbrl/companyfacts/CIK' + cik + '.json'
    response = requests.get(url,headers=headers)
    response.raise_for_status()
    try:
        response = response.json()['facts']['us-gaap']['InventoryNet']['units']['USD']
    except:
        response = None
    return response


In [ ]:
data = pd.DataFrame(columns=['company', 'industry', 'inventory'])
for i in range(len(companies)):
    symbol = companies['Symbol'][i]
    sec_data = fetch_sec_data(companies['CIK'][i])
    if sec_data is not None:
        data.loc[len(data)] = [symbol, companies['GICS Sub-Industry'][i], sec_data]

data.head()

0001467373
0000796343
0000004977
0001559720
0001086222
0001035443
0000352541
0000899051
0000004904
0000004962
0000005272
0001053507
0001410636
0000820027
0000315293
0001858681
0000947484
0000354190
0001267238
0000731802
0000769397
0000008670
0000866787
0000915912
0000070858
0002012383
0001393818
0001390777
0000012927
0001075531
0000079282
0001037540
0001043277
0000906345
0000927628
0001374310
0001138118
0001071739
0000316709
0001091667
0000896159
0000020286
0000831001
0000759944
0001156375
0000811156
0001058290
0001679788
0001166691
0001047862
0001175454
0001535527
0001051470
0000277948
0001561550
0001725057
0000027904
0001297996
0000935703
0001792789
0000882184
0000936340
0001065088
0001156039
0000065984
0001352010
0000033213
0000033185
0001101239
0000906107
0000922621
0000920522
0001095073
0001711269
0000072741
0001109357
0001324424
0000746515
0001289490
0001013237
0000814547
0000034903
0001048911
0001136893
0000035527
0000798354
0000038777
0000749251
0000320335
0001609711
0000886982

,company,industry,inventory
0,MMM,Industrial Conglomerates,"[{'end': '2008-12-31', 'val': 3013000000, 'acc..."
1,AOS,Building Products,"[{'end': '2009-12-31', 'val': 215100000, 'accn..."
2,ABT,Health Care Equipment,"[{'end': '2007-12-31', 'val': 2951442000, 'acc..."
3,ABBV,Biotechnology,"[{'end': '2012-12-31', 'val': 1091000000, 'acc..."
4,AMD,Semiconductors,"[{'end': '2009-12-26', 'val': 567000000, 'accn..."


Export to a csv file for easy use later

In [ ]:
data.to_csv('data/inventory_data.csv', index=False)

### Data Exploration

In [2]:
inventory = pd.read_csv('data/inventory_data.csv')

Observe the shape of the data

In [3]:
print(inventory.shape)

(322, 3)


Describe the data

In [4]:
print(inventory.describe())

       company               industry  \
count      322                    322   
unique     322                     98   
top        MMM  Health Care Equipment   
freq         1                     17   

                                                inventory  
count                                                 322  
unique                                                319  
top     [{'end': '2015-12-31', 'val': 491000000, 'accn...  
freq                                                    2  


Example of the data

In [5]:
inventory.head(5)

,company,industry,inventory
0,MMM,Industrial Conglomerates,"[{'end': '2008-12-31', 'val': 3013000000, 'acc..."
1,AOS,Building Products,"[{'end': '2009-12-31', 'val': 215100000, 'accn..."
2,ABT,Health Care Equipment,"[{'end': '2007-12-31', 'val': 2951442000, 'acc..."
3,ABBV,Biotechnology,"[{'end': '2012-12-31', 'val': 1091000000, 'acc..."
4,AMD,Semiconductors,"[{'end': '2009-12-26', 'val': 567000000, 'accn..."


Check for nulls

In [6]:
inventory.isnull().sum()

company      0
industry     0
inventory    0
dtype: int64

Check for duplicates

In [7]:
inventory.duplicated().value_counts()

False    322
Name: count, dtype: int64

### Data Cleaning

Remove duplicate & unnecessary information from inventory column

In [8]:
def clean_inventory(data): 
    data_list = ast.literal_eval(data)
    date = ''
    new_data = [] 
    for dict in data_list:
        if dict['end'] != date: 
            date = dict['end'] 
            temp = {'end' : date, 'inventory' : dict['val']}
            new_data.append(temp) 
    return new_data

In [10]:
inventory['inventory'] = inventory['inventory'].apply(clean_inventory)
inventory.head(5)

,company,industry,inventory
0,MMM,Industrial Conglomerates,"[{'end': '2008-12-31', 'inventory': 3013000000..."
1,AOS,Building Products,"[{'end': '2009-12-31', 'inventory': 215100000}..."
2,ABT,Health Care Equipment,"[{'end': '2007-12-31', 'inventory': 2951442000..."
3,ABBV,Biotechnology,"[{'end': '2012-12-31', 'inventory': 1091000000..."
4,AMD,Semiconductors,"[{'end': '2009-12-26', 'inventory': 567000000}..."


Create separate date and inventory columns

In [9]:
def split_data(row):
    df = pd.DataFrame(columns=['company','industry', 'inventory', 'date'])
    for dict in row['inventory']:
        df.loc[len(df)] = [row['company'], row['industry'], dict['inventory'], pd.to_datetime(dict['end'])]
    return df

In [11]:
clean_inventory = pd.DataFrame(columns = ['company', 'industry', 'inventory', 'date'])
for i in range(len(inventory)):
    clean_inventory = pd.concat([clean_inventory, split_data(inventory.iloc[i])], ignore_index=True)


Briefly inspect the cleaned data

In [12]:
print(len(clean_inventory))

17530


In [13]:
clean_inventory.tail()

,company,industry,inventory,date
17525,ZTS,Pharmaceuticals,2306000000,2024-12-31 00:00:00
17526,ZTS,Pharmaceuticals,2365000000,2025-03-31 00:00:00
17527,ZTS,Pharmaceuticals,2439000000,2025-06-30 00:00:00
17528,ZTS,Pharmaceuticals,2465000000,2025-09-30 00:00:00
17529,ZTS,Pharmaceuticals,2430000000,2025-12-31 00:00:00


### Data Aquisition pt 2

Use the current data to grab stock data

In [14]:
def acquire_stock_data(row):
    ticker = row['company']
    for i in range(5): # in case date is on weekend of holiday, loop back to first available date
        date = pd.to_datetime(row['date'])
        interval = pd.Timedelta(days=i)
        try:
            stock = yf.download(tickers=ticker, start=date-interval, end=date-interval+pd.Timedelta(days=1))['Close'][ticker][date]#[str(row['date'].date())]
            return stock
        except:
            pass
    return None 


In [15]:
print(acquire_stock_data(clean_inventory.iloc[0]))

[*********************100%***********************]  1 of 1 completed

28.680505752563477


In [ ]:
stocks_data = pd.DataFrame(columns=['company', 'date', 'price'])

for i in range(10):
    stocks_data.loc[len(stocks_data)] = [clean_inventory.iloc['row'], clean_inventory.iloc['date'], acquire_stock_data(clean_inventory.iloc[i])]

stocks_data.head(5)

NameError: name 'pd' is not defined